# DATASCAPE 2026 — final submission ConvNeXt-Small 384 (T4)

Notebook ini melatih satu model final pada seluruh **12.224 gambar train** dengan label resmi, lalu membuat `submission.csv` untuk test set.

Konfigurasi dipilih dari eksperimen A/B sebelumnya:

- `convnext_small.fb_in22k_ft_in1k_384`;
- input 384 px;
- NVIDIA T4 + FP16 AMP;
- physical batch 32, gradient accumulation 2, effective batch 64;
- 15 epoch, AdamW, cosine decay, dan EMA;
- horizontal-flip TTA untuk prediksi test;
- **tanpa pseudo-label, flip label, atau downweight**.

Manifest hasil pipeline sebelumnya hanya dipakai untuk daftar file dan label resmi. Pilih runtime **T4 GPU**, set `KAGGLE_API_TOKEN` di Colab Secrets bila dataset belum ada, lalu jalankan **Run all**.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
    'torch': 'torch', 'torchvision': 'torchvision', 'timm': 'timm>=1.0.20',
    'tqdm': 'tqdm', 'kagglehub': 'kagglehub',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import contextlib
import gc
import hashlib
import json
import math
import os
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from timm.utils import ModelEmaV3
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    raise RuntimeError('Notebook ini dikonfigurasi untuk Google Colab T4.')

from google.colab import drive
drive.mount('/content/drive')
print('Python:', sys.version.split()[0], '| torch:', torch.__version__, '| timm:', timm.__version__)


In [ ]:
SEED = 42
CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
INDEX_TO_CLASS = {index: label for label, index in CLASS_TO_INDEX.items()}
COMPETITION = 'datascape-2026-data-mining-competition'
MODEL_ID = 'hf_hub:timm/convnext_small.fb_in22k_ft_in1k_384'
IMAGE_SIZE = 384
FIXED_EPOCHS = 15
PHYSICAL_BATCH_SIZE = 32
TARGET_EFFECTIVE_BATCH = 64
GRAD_ACCUM_STEPS = TARGET_EFFECTIVE_BATCH // PHYSICAL_BATCH_SIZE
EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * GRAD_ACCUM_STEPS
EVAL_BATCH_SIZE = 64
REFERENCE_BATCH = 128
REFERENCE_LR = 2e-4
BASE_LR = REFERENCE_LR * math.sqrt(EFFECTIVE_BATCH_SIZE / REFERENCE_BATCH)
MIN_LR = 1e-6
WARMUP_EPOCHS = 1
WEIGHT_DECAY = 5e-2
LABEL_SMOOTHING = 0.08
BALANCE_STRENGTH = 0.50
GRAD_CLIP_NORM = 1.0
DROP_RATE = 0.20
DROP_PATH_RATE = 0.20
MODEL_EMA_DECAY = 0.9997
USE_HORIZONTAL_FLIP_TTA = True
NUM_WORKERS = min(4, os.cpu_count() or 2)
PIN_MEMORY = True

DRIVE_ROOT = Path('/content/drive/MyDrive/DATASCAPE')
MANIFEST_PATH = DRIVE_ROOT / 'siglip2_first_pipeline/siglip2_grouped_manifest.csv'
EXPECTED_MANIFEST_SHA256 = '34d911482f95de94c4a854cac927b5ba5d7aa3ba6e55973f7d1201825563a7d2'
OUTPUT_DIR = DRIVE_ROOT / 'submission_convnext_small_384_t4_original_e15_eb64'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_ROOT = Path('/content') / COMPETITION
DOWNLOAD_ROOT = Path('/content/datascape_dataset')

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak aktif. Pilih Runtime > Change runtime type > T4 GPU.')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'T4' not in GPU_NAME.upper():
    raise RuntimeError(f'Runtime guard: expected NVIDIA T4, got {GPU_NAME}.')
if TARGET_EFFECTIVE_BATCH % PHYSICAL_BATCH_SIZE:
    raise ValueError('Effective batch harus habis dibagi physical batch.')
print({
    'gpu': GPU_NAME, 'vram_gib': round(GPU_VRAM_GIB, 1), 'amp': 'FP16',
    'image_size': IMAGE_SIZE, 'physical_batch': PHYSICAL_BATCH_SIZE,
    'gradient_accumulation': GRAD_ACCUM_STEPS,
    'effective_batch': EFFECTIVE_BATCH_SIZE, 'base_lr': BASE_LR,
    'epochs': FIXED_EPOCHS, 'output': str(OUTPUT_DIR),
})


In [ ]:
def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def safe_extract(zip_path, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    return destination


def find_dataset_root(start):
    start = Path(start)
    candidates = [start]
    if start.exists():
        candidates.extend(path.parent for path in start.rglob('sample_submission.csv'))
    for root in candidates:
        if (root / 'sample_submission.csv').is_file() and (root / 'Train_Set').is_dir() and (root / 'Test_Set').is_dir():
            if all((root / 'Train_Set' / label).is_dir() for label in CLASSES):
                return root
    raise FileNotFoundError(start)


def resolve_dataset_root():
    for candidate in (LOCAL_DATA_ROOT, DOWNLOAD_ROOT):
        try:
            return find_dataset_root(candidate)
        except FileNotFoundError:
            pass
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    try:
        from google.colab import userdata
        token = (userdata.get('KAGGLE_API_TOKEN') or token).strip()
    except Exception:
        pass
    if not token:
        raise RuntimeError('Dataset belum ada dan KAGGLE_API_TOKEN tidak ditemukan di Colab Secrets.')
    import kagglehub
    from kagglehub.config import set_kaggle_api_token
    set_kaggle_api_token(token)
    downloaded = Path(kagglehub.competition_download(COMPETITION))
    if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
        return find_dataset_root(safe_extract(downloaded, DOWNLOAD_ROOT))
    return find_dataset_root(downloaded)

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'Manifest tidak ditemukan: {MANIFEST_PATH}')
if sha256_file(MANIFEST_PATH) != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('Hash manifest berbeda dari manifest yang dievaluasi pada eksperimen A/B.')

DATA_ROOT = resolve_dataset_root()
manifest = pd.read_csv(MANIFEST_PATH)
required_columns = {'filepath', 'label', 'dedup_group', 'partition'}
missing_columns = required_columns - set(manifest.columns)
if missing_columns:
    raise ValueError(f'Kolom manifest kurang: {sorted(missing_columns)}')
if len(manifest) != 12224:
    raise ValueError(f'Jumlah train seharusnya 12224, ditemukan {len(manifest)}')
if not set(manifest.label).issubset(CLASSES):
    raise ValueError(f'Label tidak dikenal: {sorted(set(manifest.label) - set(CLASSES))}')

train_frame = manifest[['filepath', 'label', 'dedup_group']].copy()
train_frame = train_frame.rename(columns={'label': 'label_original'})
train_frame['image_path'] = train_frame.filepath.map(lambda value: str(DATA_ROOT / value))
missing_train = train_frame.loc[~train_frame.image_path.map(lambda value: Path(value).is_file()), 'filepath']
if len(missing_train):
    raise FileNotFoundError(f'{len(missing_train)} gambar train hilang; contoh: {missing_train.iloc[0]}')

# Selalu ikuti urutan sample_submission.
template = pd.read_csv(DATA_ROOT / 'sample_submission.csv', dtype={'filename': str})
if not {'filename', 'label'}.issubset(template.columns):
    raise ValueError(f'Format sample_submission tidak sesuai: {list(template.columns)}')
test_files = [path for path in (DATA_ROOT / 'Test_Set').rglob('*') if path.is_file()]
if len({path.name for path in test_files}) != len(test_files):
    raise ValueError('Ada nama file test duplikat di subfolder berbeda.')
by_name = {path.name: path for path in test_files}
missing_test = [name for name in template.filename if name not in by_name]
if missing_test:
    raise FileNotFoundError(f'{len(missing_test)} gambar test hilang; contoh: {missing_test[0]}')
test_frame = pd.DataFrame({
    'filename': template.filename.astype(str),
    'image_path': [str(by_name[name]) for name in template.filename.astype(str)],
})

print('Dataset root:', DATA_ROOT)
print('Train full resmi:', len(train_frame), train_frame.label_original.value_counts().to_dict())
print('Test:', len(test_frame))
print('Pseudo-label digunakan: 0')


In [ ]:
fill = tuple(int(round(value * 255)) for value in IMAGENET_MEAN)

class Letterbox:
    def __init__(self, size):
        self.size = int(size)
    def __call__(self, image):
        return ImageOps.pad(
            image, (self.size, self.size), Image.Resampling.BILINEAR,
            color=fill, centering=(0.5, 0.5),
        )


def load_rgb_image(path):
    with Image.open(path) as opened:
        image = ImageOps.exif_transpose(opened)
        if image.mode == 'P':
            image = image.convert('RGBA')
        if image.mode in {'RGBA', 'LA'} or 'transparency' in image.info:
            rgba = image.convert('RGBA')
            rgba_fill = tuple(int(value * 255) for value in IMAGENET_MEAN) + (255,)
            image = Image.alpha_composite(Image.new('RGBA', rgba.size, rgba_fill), rgba).convert('RGB')
        else:
            image = image.convert('RGB')
        return image.copy()

TRAIN_TRANSFORM = transforms.Compose([
    Letterbox(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomAffine(
        degrees=3.0, translate=(0.02, 0.02), scale=(0.97, 1.03),
        interpolation=transforms.InterpolationMode.BILINEAR, fill=fill,
    ),
    transforms.ColorJitter(brightness=0.06, contrast=0.06, saturation=0.06, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
EVAL_TRANSFORM = transforms.Compose([
    Letterbox(IMAGE_SIZE), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class TrainDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return TRAIN_TRANSFORM(load_rgb_image(row.image_path)), CLASS_TO_INDEX[row.label_original]

class TestDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        return EVAL_TRANSFORM(load_rgb_image(self.frame.iloc[index].image_path)), index


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_train_loader(frame):
    counts = frame.label_original.value_counts()
    weights = frame.label_original.map(lambda label: counts[label] ** -BALANCE_STRENGTH).to_numpy()
    sampler = WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.double), len(frame), replacement=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    return DataLoader(
        TrainDataset(frame), batch_size=PHYSICAL_BATCH_SIZE, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED + 1),
    )


def make_test_loader(frame):
    return DataLoader(
        TestDataset(frame), batch_size=EVAL_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )

print('Train batches:', math.ceil(len(train_frame) / PHYSICAL_BATCH_SIZE), '| workers:', NUM_WORKERS)


In [ ]:
def autocast_context():
    return torch.autocast('cuda', dtype=torch.float16)


def build_model():
    return timm.create_model(
        MODEL_ID, pretrained=True, num_classes=len(CLASSES),
        drop_rate=DROP_RATE, drop_path_rate=DROP_PATH_RATE,
    ).cuda(memory_format=torch.channels_last)


def verify_memory_fit():
    model = optimizer = scaler = images = targets = None
    try:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        model = build_model()
        optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
        scaler = torch.amp.GradScaler('cuda', enabled=True)
        images = torch.randn(PHYSICAL_BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device='cuda').contiguous(
            memory_format=torch.channels_last
        )
        targets = torch.randint(0, len(CLASSES), (PHYSICAL_BATCH_SIZE,), device='cuda')
        with autocast_context():
            loss = F.cross_entropy(model(images), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()
        reserved = torch.cuda.max_memory_reserved() / torch.cuda.get_device_properties(0).total_memory
        print(f'Physical batch {PHYSICAL_BATCH_SIZE}: peak reserved {reserved:.1%}')
        if reserved > 0.95:
            raise RuntimeError(f'Batch {PHYSICAL_BATCH_SIZE} memakai {reserved:.1%} VRAM; hentikan agar tidak OOM saat training.')
    finally:
        del model, optimizer, scaler, images, targets
        gc.collect()
        torch.cuda.empty_cache()

verify_memory_fit()


In [ ]:
def set_run_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def learning_rate_at(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(FIXED_EPOCHS - WARMUP_EPOCHS - 1, 1)
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * min(progress, 1)))


def checkpoint_contract():
    return {
        'model_id': MODEL_ID,
        'image_size': IMAGE_SIZE,
        'seed': SEED,
        'fixed_epochs': FIXED_EPOCHS,
        'manifest_sha256': EXPECTED_MANIFEST_SHA256,
        'training_rows': int(len(train_frame)),
        'label_source': 'official_original',
        'pseudo_labels_used': False,
        'physical_batch': PHYSICAL_BATCH_SIZE,
        'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
        'effective_batch': EFFECTIVE_BATCH_SIZE,
    }


def train_final_model():
    checkpoint_path = OUTPUT_DIR / 'final_ema.pt'
    complete_path = OUTPUT_DIR / 'complete.json'
    history_path = OUTPUT_DIR / 'history.csv'
    contract = checkpoint_contract()
    if checkpoint_path.is_file() and complete_path.is_file():
        saved_contract = json.loads(complete_path.read_text())
        if saved_contract != contract:
            raise RuntimeError(f'Checkpoint contract mismatch: {saved_contract}')
        print('Final checkpoint sudah lengkap:', checkpoint_path)
        return checkpoint_path

    set_run_seed(SEED)
    loader = make_train_loader(train_frame)
    model = build_model()
    ema = ModelEmaV3(model, decay=MODEL_EMA_DECAY, use_warmup=True, warmup_gamma=1.0, warmup_power=2/3)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    history = []
    optimizer_updates = 0

    for epoch in range(FIXED_EPOCHS):
        lr = learning_rate_at(epoch)
        for group in optimizer.param_groups:
            group['lr'] = lr
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        total_examples = 0
        started = time.time()
        number_of_batches = len(loader)
        progress = tqdm(loader, desc=f'final epoch {epoch + 1}/{FIXED_EPOCHS}')
        for step, (images, targets) in enumerate(progress):
            images = images.cuda(non_blocking=True).contiguous(memory_format=torch.channels_last)
            targets = targets.cuda(non_blocking=True)
            window_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            window_size = min(GRAD_ACCUM_STEPS, number_of_batches - window_start)
            with autocast_context():
                raw_loss = F.cross_entropy(model(images), targets, label_smoothing=LABEL_SMOOTHING)
                loss = raw_loss / window_size
            scaler.scale(loss).backward()
            should_step = (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == number_of_batches
            if should_step:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                optimizer_updates += 1
                ema.update(model, step=optimizer_updates)
            total_loss += float(raw_loss.item()) * len(targets)
            total_examples += len(targets)
            progress.set_postfix(loss=f'{total_loss / total_examples:.4f}', lr=f'{lr:.2e}')

        row = {
            'epoch': epoch + 1,
            'lr': lr,
            'train_loss': total_loss / total_examples,
            'optimizer_updates': optimizer_updates,
            'seconds': time.time() - started,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(row)

    temporary = checkpoint_path.with_suffix('.pt.tmp')
    torch.save({'model_state': {key: value.cpu() for key, value in ema.module.state_dict().items()}}, temporary)
    temporary.replace(checkpoint_path)
    temporary_contract = complete_path.with_suffix('.json.tmp')
    temporary_contract.write_text(json.dumps(contract, indent=2) + '\n')
    temporary_contract.replace(complete_path)
    del model, ema, optimizer, scaler, loader
    gc.collect()
    torch.cuda.empty_cache()
    print('Training selesai:', checkpoint_path)
    return checkpoint_path

FINAL_CHECKPOINT = train_final_model()


In [ ]:
@torch.inference_mode()
def predict_test(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    model = build_model()
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    loader = make_test_loader(test_frame)
    chunks = []
    for images, _ in tqdm(loader, desc='predict test'):
        images = images.cuda(non_blocking=True).contiguous(memory_format=torch.channels_last)
        with autocast_context():
            probabilities = model(images).softmax(1)
            if USE_HORIZONTAL_FLIP_TTA:
                probabilities = 0.5 * (
                    probabilities + model(torch.flip(images, dims=[3])).softmax(1)
                )
        chunks.append(probabilities.float().cpu().numpy())
    result = np.concatenate(chunks)
    del model, loader, checkpoint
    gc.collect()
    torch.cuda.empty_cache()
    return result

probabilities = predict_test(FINAL_CHECKPOINT)
if probabilities.shape != (len(template), len(CLASSES)) or not np.isfinite(probabilities).all():
    raise RuntimeError(f'Probability output tidak valid: {probabilities.shape}')
np.save(OUTPUT_DIR / 'test_probabilities.npy', probabilities)

submission = template.copy()
submission['label'] = [INDEX_TO_CLASS[index] for index in probabilities.argmax(1)]
if len(submission) != len(template) or not submission.filename.equals(template.filename):
    raise RuntimeError('Urutan submission berubah.')
if submission.label.isna().any() or not set(submission.label).issubset(CLASSES):
    raise RuntimeError('Label submission tidak valid.')

submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
probability_frame = pd.DataFrame(probabilities, columns=[f'prob_{label}' for label in CLASSES])
probability_frame.insert(0, 'filename', template.filename)
probability_frame.to_csv(OUTPUT_DIR / 'test_probabilities.csv', index=False)
print(submission.label.value_counts().reindex(CLASSES, fill_value=0))
print(submission.head())
print('Submission:', submission_path)


In [ ]:
summary = {
    'protocol': 'final_submission_convnext_small_384_t4_original_labels_e15_eb64',
    'model_id': MODEL_ID,
    'image_size': IMAGE_SIZE,
    'gpu': GPU_NAME,
    'amp': 'float16_grad_scaler',
    'physical_batch': PHYSICAL_BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
    'effective_batch': EFFECTIVE_BATCH_SIZE,
    'fixed_epochs': FIXED_EPOCHS,
    'training_rows': int(len(train_frame)),
    'label_source': 'official_original_from_grouped_manifest',
    'pseudo_labels_used': False,
    'manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'test_rows': int(len(test_frame)),
    'flip_tta': USE_HORIZONTAL_FLIP_TTA,
    'submission': 'submission.csv',
    'prediction_counts': {label: int((submission.label == label).sum()) for label in CLASSES},
}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False) + '\n')

bundle_path = Path('/content/submission_convnext_small_384_t4_original_e15_eb64.zip')
result_names = [
    'submission.csv', 'test_probabilities.csv', 'test_probabilities.npy',
    'history.csv', 'run_summary.json', 'complete.json',
]
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in result_names:
        path = OUTPUT_DIR / name
        if path.is_file():
            archive.write(path, arcname=name)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Bundle:', bundle_path, '| bytes:', bundle_path.stat().st_size)
try:
    from google.colab import files
    files.download(str(bundle_path))
except Exception as error:
    print('Automatic download unavailable:', error)
